In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

SPATIAL_PATH = os.path.join(PROJECT_PATH, "spatial_features")
TEMPORAL_PATH = os.path.join(PROJECT_PATH, "temporal_features")

OUTPUT_PATH = os.path.join(PROJECT_PATH, "fusion_features")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "original"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Deepfakes"), exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
sample_file = sorted(
    os.listdir(
        os.path.join(SPATIAL_PATH, "original")
    )
)[0]

spatial_feature = np.load(
    os.path.join(
        SPATIAL_PATH,
        "original",
        sample_file
    )
)

print(spatial_feature.shape)

In [ ]:
temporal_feature = np.load(
    os.path.join(
        TEMPORAL_PATH,
        "original",
        sample_file
    )
)

print(temporal_feature.shape)

In [ ]:
spatial_feature = torch.tensor(
    spatial_feature,
    dtype=torch.float32
).unsqueeze(0).to(device)

temporal_feature = torch.tensor(
    temporal_feature,
    dtype=torch.float32
).unsqueeze(0).to(device)

print(spatial_feature.shape)
print(temporal_feature.shape)

In [ ]:
class AdaptiveCrossAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):
        super().__init__()

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, spatial, temporal):

        # Query = Spatial
        # Key = Temporal
        # Value = Temporal

        output, attention_weights = self.cross_attention(
            query=spatial,
            key=temporal,
            value=temporal
        )

        return output, attention_weights

In [ ]:
cross_attention = AdaptiveCrossAttention().to(device)

cross_attention.eval()

print(cross_attention)

In [ ]:
with torch.no_grad():

    fused_feature, attention = cross_attention(
        spatial_feature,
        temporal_feature
    )

print(fused_feature.shape)
print(attention.shape)

In [ ]:
class AdaptiveGate(nn.Module):

    def __init__(self, embed_dim=512):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

    def forward(self, features):

        gate_weights = self.gate(features)

        fused = features * gate_weights

        return fused

In [ ]:
gate = AdaptiveGate().to(device)

gate.eval()

print(gate)

In [ ]:
with torch.no_grad():

    gated_feature = gate(fused_feature)

print(gated_feature.shape)

In [ ]:
final_feature = gated_feature.mean(dim=1)

print(final_feature.shape)

In [ ]:
save_path = os.path.join(
    OUTPUT_PATH,
    "original",
    sample_file
)

np.save(
    save_path,
    final_feature.squeeze(0).cpu().numpy()
)

print("Saved Successfully!")
print(save_path)

In [ ]:
saved_feature = np.load(save_path)

print(saved_feature.shape)

In [ ]:
def fuse_features(spatial_file, temporal_file):

    spatial = np.load(spatial_file)
    temporal = np.load(temporal_file)

    spatial = torch.tensor(
        spatial,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    temporal = torch.tensor(
        temporal,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        fused, _ = cross_attention(
            spatial,
            temporal
        )

        fused = gate(fused)

        fused = fused.mean(dim=1)

    return fused.squeeze(0).cpu().numpy()

In [ ]:
feature = fuse_features(
    os.path.join(
        SPATIAL_PATH,
        "original",
        sample_file
    ),
    os.path.join(
        TEMPORAL_PATH,
        "original",
        sample_file
    )
)

print(feature.shape)

In [ ]:
def process_fusion_dataset(input_class):

    spatial_dir = os.path.join(
        SPATIAL_PATH,
        input_class
    )

    temporal_dir = os.path.join(
        TEMPORAL_PATH,
        input_class
    )

    output_dir = os.path.join(
        OUTPUT_PATH,
        input_class
    )

    os.makedirs(output_dir, exist_ok=True)

    files = sorted(os.listdir(spatial_dir))

    for file in tqdm(files):

        spatial_file = os.path.join(spatial_dir, file)
        temporal_file = os.path.join(temporal_dir, file)

        feature = fuse_features(
            spatial_file,
            temporal_file
        )

        np.save(
            os.path.join(output_dir, file),
            feature
        )

    print(f"{input_class} Completed!")

In [ ]:
process_fusion_dataset("original")

In [ ]:
process_fusion_dataset("Deepfakes")

In [ ]:
original = os.listdir(
    os.path.join(
        OUTPUT_PATH,
        "original"
    )
)

deepfake = os.listdir(
    os.path.join(
        OUTPUT_PATH,
        "Deepfakes"
    )
)

print("Original :", len(original))
print("Deepfake :", len(deepfake))

In [ ]:
sample = np.load(
    os.path.join(
        OUTPUT_PATH,
        "original",
        original[0]
    )
)

print(sample.shape)